In [7]:
# ============================================================
# MODULE 9 — Natural Language Processing System
# Full corrected version
#
# This module extends the campus project by allowing a user to
# select a campus agent and ask natural-language questions
# about that agent's current state and likely next behavior.
#
# This version includes:
# - strict parsing
# - semantic frame construction
# - ambiguity handling
# - language modeling
# - classification as the NLP task
# - grounded answers tied to RL-based agent state
# ============================================================

import math
import random
import re
from dataclasses import dataclass
from collections import defaultdict, Counter
from typing import Optional, Dict, Any, List, Tuple

import numpy as np

# ============================================================
# 0. Reproducibility
# ============================================================

RANDOM_SEED = 7
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# 1. Canonical schema
# ============================================================

TIME_SLOTS = [
    "07:00-08:00",
    "08:00-09:30",
    "09:40-11:10",
    "11:20-12:50",
    "13:00-14:30",
    "15:00-15:30",
    "16:00-17:30",
    "18:00-21:00"
]

MAIN_LOCATIONS = [
    "LawrenceHall",
    "ThayerHall",
    "WestPenn",
    "StudentCenter",
    "Library",
    "VillagePark",
    "Track",
    "PlayHouse",
    "BoulevardStudios",
    "BoulevardAppartments",
    "OffCampus"
]

DAY_OF_WEEK = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
SPORTS = ["TrackAndField", "Baseball"]
STUDENT_TYPES = ["Athlete", "Copa", "Regular Student"]

CAMPUS_GRAPH = {
    "LawrenceHall": ["StudentCenter", "Library", "ThayerHall"],
    "ThayerHall": ["LawrenceHall", "WestPenn", "Library"],
    "WestPenn": ["ThayerHall", "Track", "OffCampus"],
    "StudentCenter": ["LawrenceHall", "Library", "VillagePark", "Track", "BoulevardAppartments"],
    "Library": ["LawrenceHall", "StudentCenter", "ThayerHall"],
    "VillagePark": ["StudentCenter", "Track", "PlayHouse"],
    "Track": ["VillagePark", "StudentCenter", "WestPenn", "OffCampus"],
    "PlayHouse": ["VillagePark", "StudentCenter", "BoulevardStudios"],
    "BoulevardStudios": ["PlayHouse", "BoulevardAppartments"],
    "BoulevardAppartments": ["StudentCenter", "BoulevardStudios", "OffCampus"],
    "OffCampus": ["Track", "WestPenn", "BoulevardAppartments"]
}

TRACK_SCHEDULE = {
    ("Tuesday",  "07:00-08:00"): "StudentCenter",
    ("Thursday", "07:00-08:00"): "StudentCenter",
    ("Monday",    "15:00-15:30"): "Track",
    ("Tuesday",   "15:00-15:30"): "Track",
    ("Wednesday", "15:00-15:30"): "Track",
    ("Thursday",  "15:00-15:30"): "Track",
    ("Friday",    "15:00-15:30"): "Track",
    ("Monday",    "16:00-17:30"): "Track",
    ("Tuesday",   "16:00-17:30"): "Track",
    ("Wednesday", "16:00-17:30"): "Track",
    ("Thursday",  "16:00-17:30"): "Track",
    ("Friday",    "16:00-17:30"): "Track",
}

BASEBALL_SCHEDULE = {
    ("Monday",    "13:00-14:30"): "OffCampus",
    ("Wednesday", "13:00-14:30"): "OffCampus",
    ("Monday",    "16:00-17:30"): "StudentCenter",
    ("Tuesday",   "13:00-14:30"): "StudentCenter",
    ("Thursday",  "13:00-14:30"): "StudentCenter",
    ("Friday",    "07:00-08:00"): "StudentCenter",
    ("Friday",    "08:00-09:30"): "StudentCenter",
    ("Friday",    "09:40-11:10"): "StudentCenter",
    ("Friday",    "11:20-12:50"): "StudentCenter",
    ("Friday",    "13:00-14:30"): "StudentCenter",
}

LOCATION_ALIASES = {
    "student center": "StudentCenter",
    "studentcenter": "StudentCenter",
    "center": "StudentCenter",
    "the center": "StudentCenter",
    "library": "Library",
    "track": "Track",
    "the track": "Track",
    "village park": "VillagePark",
    "villagepark": "VillagePark",
    "playhouse": "PlayHouse",
    "play house": "PlayHouse",
    "west penn": "WestPenn",
    "westpenn": "WestPenn",
    "lawrence hall": "LawrenceHall",
    "lawrencehall": "LawrenceHall",
    "thayer hall": "ThayerHall",
    "thayerhall": "ThayerHall",
    "boulevard studios": "BoulevardStudios",
    "boulevardstudios": "BoulevardStudios",
    "boulevard appartments": "BoulevardAppartments",
    "boulevard apartments": "BoulevardAppartments",
    "boulevardappartments": "BoulevardAppartments",
    "off campus": "OffCampus",
    "offcampus": "OffCampus",
}

TIME_ALIASES = {
    "7": "07:00-08:00",
    "7am": "07:00-08:00",
    "7 am": "07:00-08:00",
    "8": "08:00-09:30",
    "8am": "08:00-09:30",
    "8 am": "08:00-09:30",
    "940": "09:40-11:10",
    "9:40": "09:40-11:10",
    "11:20": "11:20-12:50",
    "1": "13:00-14:30",
    "1pm": "13:00-14:30",
    "1 pm": "13:00-14:30",
    "3": "15:00-15:30",
    "3pm": "15:00-15:30",
    "3 pm": "15:00-15:30",
    "4": "16:00-17:30",
    "4pm": "16:00-17:30",
    "4 pm": "16:00-17:30",
    "6": "18:00-21:00",
    "6pm": "18:00-21:00",
    "6 pm": "18:00-21:00",
}

SUPPORTED_INTENTS = [
    "next_destination",
    "destination_reason",
    "current_location_reason",
    "next_location_prediction",
    "validity_check",
    "probability_to_location"
]

# ============================================================
# 2. Module 8 RL environment and policy helpers
# ============================================================

class CampusRLEnvironment:
    def __init__(self, sport: str = "TrackAndField", day: str = "Tuesday"):
        self.sport = sport
        self.day = day
        self.time_slots = TIME_SLOTS
        self.locations = MAIN_LOCATIONS
        self.graph = CAMPUS_GRAPH
        self.start_location = "OffCampus"
        self.reset()

    def reset(self) -> Tuple[int, str]:
        self.t = 0
        self.location = self.start_location
        return (self.t, self.location)

    def get_target_location(self, t: int) -> Optional[str]:
        if t < 0 or t >= len(self.time_slots):
            return None

        key = (self.day, self.time_slots[t])

        if self.sport == "TrackAndField":
            return TRACK_SCHEDULE.get(key, None)
        if self.sport == "Baseball":
            return BASEBALL_SCHEDULE.get(key, None)
        return None

    def is_terminal(self, state: Tuple[int, str]) -> bool:
        t, _ = state
        return t >= len(self.time_slots) - 1

    def available_actions(self, state: Tuple[int, str]) -> List[Tuple[str, str]]:
        _, loc = state
        actions = [("STAY", loc)]
        for nxt in self.graph[loc]:
            actions.append(("MOVE", nxt))
        return actions

    def transition(self, state: Tuple[int, str], action: Tuple[str, str]) -> Tuple[int, str]:
        t, loc = state
        act_type, dest = action
        next_t = min(t + 1, len(self.time_slots) - 1)
        next_loc = loc if act_type == "STAY" else dest
        return (next_t, next_loc)

    def reward(
        self,
        state: Tuple[int, str],
        action: Tuple[str, str],
        next_state: Tuple[int, str]
    ) -> float:
        next_t, next_loc = next_state
        target = self.get_target_location(next_t)

        r = 0.0

        if action[0] == "MOVE":
            r -= 1.0

        if target is not None:
            if next_loc == target:
                r += 8.0
            else:
                r -= 3.0

        if next_loc == "StudentCenter":
            r += 0.75

        if self.sport == "TrackAndField" and next_loc == "Track":
            r += 1.0

        if self.sport == "Baseball" and next_loc == "OffCampus" and target == "OffCampus":
            r += 0.75

        if target is not None and next_loc == "OffCampus" and target != "OffCampus":
            r -= 2.0

        if next_loc == "VillagePark":
            r += 0.2

        return r

    def step(self, action: Tuple[str, str]) -> Tuple[Tuple[int, str], float, bool]:
        state = (self.t, self.location)
        next_state = self.transition(state, action)
        r = self.reward(state, action, next_state)
        self.t, self.location = next_state
        done = self.is_terminal(next_state)
        return next_state, r, done

def epsilon_greedy_action(
    env: CampusRLEnvironment,
    state: Tuple[int, str],
    Q: Dict[Tuple[Tuple[int, str], Tuple[str, str]], float],
    epsilon: float
) -> Tuple[str, str]:
    actions = env.available_actions(state)
    if random.random() < epsilon:
        return random.choice(actions)
    q_vals = [Q[(state, a)] for a in actions]
    best_idx = int(np.argmax(q_vals))
    return actions[best_idx]

def q_learning(
    env: CampusRLEnvironment,
    num_episodes: int = 3000,
    alpha: float = 0.10,
    gamma: float = 0.95,
    epsilon: float = 0.25,
    epsilon_decay: float = 0.998,
    epsilon_min: float = 0.02
) -> Dict[Tuple[Tuple[int, str], Tuple[str, str]], float]:
    Q: Dict[Tuple[Tuple[int, str], Tuple[str, str]], float] = defaultdict(float)

    for _ in range(num_episodes):
        state = env.reset()
        done = False

        while not done:
            action = epsilon_greedy_action(env, state, Q, epsilon)
            next_state, reward, done = env.step(action)

            next_actions = env.available_actions(next_state)
            max_next_q = max(Q[(next_state, a)] for a in next_actions)

            td_target = reward + gamma * max_next_q
            td_error = td_target - Q[(state, action)]
            Q[(state, action)] += alpha * td_error

            state = next_state

        epsilon = max(epsilon * epsilon_decay, epsilon_min)

    return Q

def greedy_policy_from_q(Q):
    def policy(env: CampusRLEnvironment, state: Tuple[int, str]) -> Tuple[str, str]:
        actions = env.available_actions(state)
        q_vals = [Q[(state, a)] for a in actions]
        best_idx = int(np.argmax(q_vals))
        return actions[best_idx]
    return policy

# ============================================================
# 3. Agent representation
# ============================================================

@dataclass
class AgentSnapshot:
    agent_id: int
    student_type: str
    sport: Optional[str]
    day: str
    time_index: int
    time_slot: str
    current_location: str
    next_action: Tuple[str, str]
    next_destination: str
    target_location_now: Optional[str]
    target_location_next: Optional[str]

def infer_student_type_from_sport(sport: Optional[str]) -> str:
    if sport in SPORTS:
        return "Athlete"
    return "Regular Student"

def make_agent_snapshot_from_policy(
    agent_id: int,
    sport: str,
    day: str,
    time_index: int,
    current_location: str,
    policy_fn
) -> AgentSnapshot:
    env = CampusRLEnvironment(sport=sport, day=day)
    state = (time_index, current_location)
    next_action = policy_fn(env, state)
    next_destination = next_action[1]
    target_location_now = env.get_target_location(time_index)
    target_location_next = env.get_target_location(min(time_index + 1, len(TIME_SLOTS) - 1))

    return AgentSnapshot(
        agent_id=agent_id,
        student_type=infer_student_type_from_sport(sport),
        sport=sport,
        day=day,
        time_index=time_index,
        time_slot=TIME_SLOTS[time_index],
        current_location=current_location,
        next_action=next_action,
        next_destination=next_destination,
        target_location_now=target_location_now,
        target_location_next=target_location_next
    )

# ============================================================
# 4. Part 1 — Language Modeling (23.1)
# ============================================================

LANGUAGE_MODEL_CORPUS = [
    "where are you going",
    "where will you go next",
    "where are you likely to be next",
    "why are you going there",
    "why are you headed there",
    "are you in the right place right now",
    "are you supposed to be here",
    "what is the probability you go to track",
    "what is the probability you go to student center",
    "what is the probability you go to library",
    "will you be in the center next",
    "are you going there next",
    "are you going to track",
    "where are you going next",
    "why are you here",
    "what is your most likely next location"
]

def tokenize(text: str) -> List[str]:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

class BigramLanguageModel:
    def __init__(self, corpus_sentences: List[str]):
        self.unigram_counts = Counter()
        self.bigram_counts = Counter()
        self.vocab = set()

        for sentence in corpus_sentences:
            tokens = ["<s>"] + tokenize(sentence) + ["</s>"]
            self.vocab.update(tokens)
            for tok in tokens:
                self.unigram_counts[tok] += 1
            for i in range(len(tokens) - 1):
                self.bigram_counts[(tokens[i], tokens[i + 1])] += 1

        self.vocab_size = len(self.vocab)

    def sentence_log_prob(self, sentence: str) -> float:
        tokens = ["<s>"] + tokenize(sentence) + ["</s>"]
        logp = 0.0
        for i in range(len(tokens) - 1):
            w1, w2 = tokens[i], tokens[i + 1]
            num = self.bigram_counts[(w1, w2)] + 1
            den = self.unigram_counts[w1] + self.vocab_size
            logp += math.log(num / den)
        return logp

LM = BigramLanguageModel(LANGUAGE_MODEL_CORPUS)

# ============================================================
# 5. Raw text normalization
# ============================================================

def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

# ============================================================
# 6. STRICT parsing helpers
# ============================================================

def extract_explicit_location_mentions(text: str) -> List[Tuple[str, float]]:
    q = normalize_text(text)
    matches: List[Tuple[str, float]] = []

    for alias in sorted(LOCATION_ALIASES.keys(), key=len, reverse=True):
        pattern = r"\b" + re.escape(alias) + r"\b"
        if re.search(pattern, q):
            matches.append((LOCATION_ALIASES[alias], 1.0))

    deduped: Dict[str, float] = {}
    for loc, score in matches:
        deduped[loc] = max(deduped.get(loc, 0.0), score)

    return sorted(deduped.items(), key=lambda x: x[1], reverse=True)

def extract_explicit_time_reference(text: str) -> Optional[str]:
    q = normalize_text(text)

    for alias in sorted(TIME_ALIASES.keys(), key=len, reverse=True):
        pattern = r"\b" + re.escape(alias) + r"\b"
        if re.search(pattern, q):
            return TIME_ALIASES[alias]

    return None

# ============================================================
# 7. Part 5 — NLP Task: Classification (23.6)
# ============================================================

INTENT_PATTERNS = {
    "next_destination": [
        "where are you going",
        "where are you going next",
        "where will you go",
        "are you going there next",
        "are you going to"
    ],
    "destination_reason": [
        "why are you going there",
        "why are you headed there",
        "why are you going",
        "why there"
    ],
    "current_location_reason": [
        "why are you here",
        "why are you at",
        "why are you in"
    ],
    "next_location_prediction": [
        "where are you likely to be next",
        "where will you be next",
        "what is your most likely next location",
        "will you be in"
    ],
    "validity_check": [
        "are you in the right place",
        "are you supposed to be here",
        "are you supposed to be there",
        "are you where you should be"
    ],
    "probability_to_location": [
        "what is the probability",
        "probability you go to",
        "probability you will be in",
        "what are the chances"
    ]
}

def classify_intent(question: str) -> Tuple[str, float]:
    q = normalize_text(question)
    scores = {intent: 0.0 for intent in SUPPORTED_INTENTS}

    for intent, patterns in INTENT_PATTERNS.items():
        for p in patterns:
            if p in q:
                scores[intent] += 2.0

    if "why" in q and "here" in q:
        scores["current_location_reason"] += 1.5
    elif "why" in q:
        scores["destination_reason"] += 0.5

    if "probability" in q or "chance" in q or "chances" in q:
        scores["probability_to_location"] += 1.0

    if "right place" in q or "supposed" in q or "should be" in q:
        scores["validity_check"] += 1.0

    if "going" in q and "where" in q:
        scores["next_destination"] += 0.5

    if "next" in q and ("where" in q or "be" in q):
        scores["next_location_prediction"] += 0.5

    best_intent, best_score = max(scores.items(), key=lambda x: x[1])

    if best_score <= 0:
        return "unknown", 0.0

    total = sum(max(v, 0.0) for v in scores.values()) + 1e-9
    return best_intent, best_score / total

# ============================================================
# 8. Part 2 — Grammar and Parsing (23.2–23.3)
# ============================================================

def parse_question(question: str) -> Dict[str, Any]:
    normalized = normalize_text(question)
    intent, intent_conf = classify_intent(question)
    explicit_locations = extract_explicit_location_mentions(question)
    explicit_time = extract_explicit_time_reference(question)

    if intent == "next_destination":
        grammar_match = "QUERY -> WHERE_GOING"
    elif intent == "destination_reason":
        grammar_match = "QUERY -> WHY_DESTINATION"
    elif intent == "current_location_reason":
        grammar_match = "QUERY -> WHY_CURRENT_LOCATION"
    elif intent == "next_location_prediction":
        grammar_match = "QUERY -> FUTURE_LOCATION"
    elif intent == "validity_check":
        grammar_match = "QUERY -> VALIDITY_CHECK"
    elif intent == "probability_to_location":
        grammar_match = "QUERY -> PROBABILITY_LOCATION"
    else:
        grammar_match = "QUERY -> UNKNOWN"

    primary_location = explicit_locations[0][0] if explicit_locations else None

    return {
        "raw_question": question,
        "normalized_question": normalized,
        "intent": intent,
        "intent_confidence": intent_conf,
        "grammar_match": grammar_match,
        "explicit_location_candidates": explicit_locations,
        "primary_location": primary_location,
        "explicit_time_reference": explicit_time,
        "language_model_logprob": LM.sentence_log_prob(normalized)
    }

def build_semantic_frame(parsed: Dict[str, Any], agent: AgentSnapshot) -> Dict[str, Any]:
    intent = parsed["intent"]
    location = parsed["primary_location"]

    frame = {
        "query_type": None,
        "agent_ref": "you",
        "predicate": None,
        "location": location,
        "time": None
    }

    if intent == "next_destination":
        frame["query_type"] = "NEXT_DESTINATION"
        frame["predicate"] = "go_to"
        frame["time"] = "next"
    elif intent == "destination_reason":
        frame["query_type"] = "DESTINATION_REASON"
        frame["predicate"] = "go_to"
        frame["time"] = "next"
    elif intent == "current_location_reason":
        frame["query_type"] = "CURRENT_LOCATION_REASON"
        frame["predicate"] = "be_at"
        frame["time"] = "now"
    elif intent == "next_location_prediction":
        frame["query_type"] = "FUTURE_LOCATION_CHECK"
        frame["predicate"] = "be_in"
        frame["time"] = "next"
    elif intent == "validity_check":
        frame["query_type"] = "VALIDITY_CHECK"
        frame["predicate"] = "be_at"
        frame["time"] = "now"
    elif intent == "probability_to_location":
        frame["query_type"] = "PROBABILITY_QUERY"
        frame["predicate"] = "go_to"
        frame["time"] = "next"
    else:
        frame["query_type"] = "UNKNOWN"

    return frame

# ============================================================
# 9. Part 3 — Augmented Grammar (23.4)
# ============================================================

def validate_semantics(agent: AgentSnapshot, parsed: Dict[str, Any]) -> Dict[str, Any]:
    issues: List[str] = []
    warnings: List[str] = []

    loc = parsed["primary_location"]

    if loc is not None and loc not in MAIN_LOCATIONS:
        issues.append(f"Invalid campus location: {loc}")

    if parsed["intent"] == "probability_to_location" and loc is None:
        issues.append("Probability question requires an explicit recognizable location.")

    if parsed["explicit_time_reference"] == "07:00-08:00" and agent.student_type != "Athlete":
        warnings.append("07:00-08:00 is treated as an early athletic window in the broader project.")

    return {
        "is_valid": len(issues) == 0,
        "issues": issues,
        "warnings": warnings
    }

# ============================================================
# 10. Part 4 — Handling Ambiguity (23.5)
# ============================================================

def resolve_contextual_time_reference(question: str, agent: AgentSnapshot) -> Optional[str]:
    q = normalize_text(question)

    if re.search(r"\bnow\b", q):
        return agent.time_slot
    if re.search(r"\blater\b", q):
        next_index = min(agent.time_index + 1, len(TIME_SLOTS) - 1)
        return TIME_SLOTS[next_index]

    return extract_explicit_time_reference(question)

def generate_interpretations(
    question: str,
    agent: AgentSnapshot,
    parsed: Dict[str, Any]
) -> List[Dict[str, Any]]:
    q = parsed["normalized_question"]
    interpretations: List[Dict[str, Any]] = []

    interpretations.append({
        "source": "strict_parse",
        "intent": parsed["intent"],
        "location": parsed["primary_location"],
        "time_reference": parsed["explicit_time_reference"],
        "score": 1.0 + parsed["intent_confidence"]
    })

    if re.search(r"\bthere\b", q):
        interpretations.append({
            "source": "ambiguity_resolution",
            "intent": parsed["intent"] if parsed["intent"] != "unknown" else "next_destination",
            "location": agent.next_destination,
            "time_reference": resolve_contextual_time_reference(question, agent),
            "score": 0.85
        })
        interpretations.append({
            "source": "ambiguity_resolution",
            "intent": parsed["intent"] if parsed["intent"] != "unknown" else "validity_check",
            "location": agent.current_location,
            "time_reference": resolve_contextual_time_reference(question, agent),
            "score": 0.70
        })

    if re.search(r"\bhere\b", q):
        here_intent = parsed["intent"] if parsed["intent"] != "unknown" else "current_location_reason"
        interpretations.append({
            "source": "ambiguity_resolution",
            "intent": here_intent,
            "location": agent.current_location,
            "time_reference": resolve_contextual_time_reference(question, agent),
            "score": 0.85
        })

    if re.search(r"\b4\b", q) and "4pm" not in q and "4 pm" not in q:
        interpretations.append({
            "source": "ambiguity_resolution",
            "intent": parsed["intent"],
            "location": parsed["primary_location"],
            "time_reference": "16:00-17:30",
            "score": 0.80
        })

    best: Dict[Tuple[Any, Any, Any, Any], Dict[str, Any]] = {}
    for item in interpretations:
        key = (
            item.get("source"),
            item.get("intent"),
            item.get("location"),
            item.get("time_reference")
        )
        if key not in best or item["score"] > best[key]["score"]:
            best[key] = item

    return sorted(best.values(), key=lambda x: x["score"], reverse=True)

# ============================================================
# 11. Answer logic
# ============================================================

def choose_best_interpretation(
    parsed: Dict[str, Any],
    interpretations: List[Dict[str, Any]]
) -> Dict[str, Any]:
    # For now, strict parse remains primary.
    return interpretations[0]

def estimate_probability_to_location(agent: AgentSnapshot, location: str) -> float:
    score = 0.05

    if location == agent.next_destination:
        score += 0.60
    if location == agent.current_location:
        score += 0.20
    if location == agent.target_location_now:
        score += 0.15
    if location in CAMPUS_GRAPH.get(agent.current_location, []):
        score += 0.10

    return min(score, 0.95)

def generate_answer(
    agent: AgentSnapshot,
    parsed: Dict[str, Any],
    frame: Dict[str, Any],
    best_interpretation: Dict[str, Any],
    semantic_check: Dict[str, Any]
) -> str:
    query_type = frame["query_type"]
    loc = best_interpretation.get("location")

    if not semantic_check["is_valid"] and parsed["intent"] == "probability_to_location":
        return "I could not answer that cleanly because: " + "; ".join(semantic_check["issues"])

    prefix = ""
    if best_interpretation["source"] == "ambiguity_resolution":
        prefix = "I interpreted your vague reference using the current agent context. "

    if query_type == "NEXT_DESTINATION":
        return (
            prefix +
            f"I am going to {agent.next_destination}. "
            f"My next action is {agent.next_action[0].lower()} from "
            f"{agent.current_location} during {agent.time_slot}."
        )

    if query_type == "DESTINATION_REASON":
        if agent.target_location_now is not None:
            return (
                prefix +
                f"I am heading to {agent.next_destination} because for "
                f"{agent.day} at {agent.time_slot}, my current target is "
                f"{agent.target_location_now}. My policy recommends "
                f"{agent.next_action[0].lower()} toward that direction."
            )
        return (
            prefix +
            f"I am heading to {agent.next_destination} because given my current "
            f"state, that is the best next move according to the policy."
        )

    if query_type == "CURRENT_LOCATION_REASON":
        if agent.current_location == agent.target_location_now:
            return (
                prefix +
                f"I am here at {agent.current_location} because this is my current "
                f"target location for {agent.time_slot} on {agent.day}."
            )
        if agent.next_destination != agent.current_location:
            return (
                prefix +
                f"I am currently at {agent.current_location} because that is my present "
                f"state before moving next. My policy then sends me toward {agent.next_destination}."
            )
        return (
            prefix +
            f"I am at {agent.current_location} because that is both my current state "
            f"and my policy's selected place to remain in for now."
        )

    if query_type == "FUTURE_LOCATION_CHECK":
        if loc is not None:
            if loc == agent.next_destination:
                return (
                    prefix +
                    f"Yes. I will most likely be in {loc} next based on my current policy."
                )
            return (
                prefix +
                f"No. My most likely next location is {agent.next_destination}, not {loc}."
            )

        return (
            prefix +
            f"I will most likely move to {agent.next_destination} next based on my current policy."
        )

    if query_type == "VALIDITY_CHECK":
        if agent.target_location_now is None:
            return (
                prefix +
                f"There is no strict scheduled target for me at {agent.time_slot}, "
                f"so being at {agent.current_location} is not automatically wrong."
            )
        if agent.current_location == agent.target_location_now:
            return (
                prefix +
                f"Yes. I am currently in the right place: {agent.current_location}."
            )
        return (
            prefix +
            f"Not exactly. I am at {agent.current_location}, but my current target "
            f"for this time is {agent.target_location_now}."
        )

    if query_type == "PROBABILITY_QUERY":
        if loc is None:
            return (
                prefix +
                "I understood this as a probability question, but I could not find "
                "an explicit valid location in your question."
            )
        prob = estimate_probability_to_location(agent, loc)
        return (
            prefix +
            f"The heuristic estimated probability that I go to {loc} next is {prob:.2f}. "
            f"This estimate is based on my current location, target, and policy-selected next move."
        )

    return (
        "I could not classify that question yet. Try asking one of these: "
        "'Where are you going?', 'Why are you going there?', "
        "'Why are you here?', "
        "'Are you in the right place right now?', "
        "'Where are you likely to be next?', or "
        "'What is the probability you go to Track?'"
    )

# ============================================================
# 12. Full NLP pipeline
# ============================================================

def answer_agent_question(agent: AgentSnapshot, question: str) -> Dict[str, Any]:
    parsed = parse_question(question)
    semantic_frame = build_semantic_frame(parsed, agent)
    semantic_check = validate_semantics(agent, parsed)
    interpretations = generate_interpretations(question, agent, parsed)
    best_interpretation = choose_best_interpretation(parsed, interpretations)
    answer = generate_answer(agent, parsed, semantic_frame, best_interpretation, semantic_check)

    return {
        "agent_id": agent.agent_id,
        "agent_state": {
            "student_type": agent.student_type,
            "sport": agent.sport,
            "day": agent.day,
            "time_slot": agent.time_slot,
            "current_location": agent.current_location,
            "next_destination": agent.next_destination,
            "target_location_now": agent.target_location_now,
        },
        "parsed_question": parsed,
        "semantic_frame": semantic_frame,
        "semantic_check": semantic_check,
        "interpretations": interpretations,
        "best_interpretation": best_interpretation,
        "answer": answer
    }

# ============================================================
# 13. Demo setup
# ============================================================

print("\n" + "=" * 70)
print("MODULE 9 — NATURAL LANGUAGE PROCESSING SYSTEM")
print("=" * 70)

print("\nTraining policy from Module 8 environment...")
env_train = CampusRLEnvironment(sport="TrackAndField", day="Tuesday")
Q = q_learning(
    env_train,
    num_episodes=3000,
    alpha=0.10,
    gamma=0.95,
    epsilon=0.25,
    epsilon_decay=0.998,
    epsilon_min=0.02
)
learned_policy = greedy_policy_from_q(Q)
print("Policy training complete.")

agents = [
    make_agent_snapshot_from_policy(
        agent_id=1,
        sport="TrackAndField",
        day="Tuesday",
        time_index=5,
        current_location="StudentCenter",
        policy_fn=learned_policy
    ),
    make_agent_snapshot_from_policy(
        agent_id=2,
        sport="TrackAndField",
        day="Tuesday",
        time_index=6,
        current_location="VillagePark",
        policy_fn=learned_policy
    ),
    make_agent_snapshot_from_policy(
        agent_id=3,
        sport="Baseball",
        day="Monday",
        time_index=4,
        current_location="WestPenn",
        policy_fn=learned_policy
    )
]

print("\nExample agent snapshots:")
for agent in agents:
    print({
        "agent_id": agent.agent_id,
        "student_type": agent.student_type,
        "sport": agent.sport,
        "day": agent.day,
        "time_slot": agent.time_slot,
        "current_location": agent.current_location,
        "next_destination": agent.next_destination,
        "target_location_now": agent.target_location_now
    })

test_questions = [
    "Where are you going?",
    "Why are you going there?",
    "Where are you likely to be next?",
    "Are you in the right place right now?",
    "Are you supposed to be here?",
    "What is the probability you go to Track?",
    "What is the probability you go to the student center?",
    "Will you be in the center next?",
    "Are you going there next?",
    "Why are you here?",
    "What is the probability you go to Library?",
    "Where will you go next?"
]

print("\n" + "=" * 70)
print("DEMO QUERIES")
print("=" * 70)

selected_agent = agents[0]

for q in test_questions:
    result = answer_agent_question(selected_agent, q)
    print("\nQuestion:", q)
    print("Parsed:", result["parsed_question"])
    print("Semantic frame:", result["semantic_frame"])
    print("Semantic check:", result["semantic_check"])
    print("Top interpretations:", result["interpretations"][:3])
    print("Best interpretation:", result["best_interpretation"])
    print("Answer:", result["answer"])

# ============================================================
# 14. Interactive CLI demo
# ============================================================

def choose_agent_cli(agent_list: List[AgentSnapshot]) -> AgentSnapshot:
    print("\nAvailable agents:")
    for agent in agent_list:
        print(
            f"[{agent.agent_id}] {agent.student_type} / {agent.sport} | "
            f"{agent.day} {agent.time_slot} | current={agent.current_location} | "
            f"next={agent.next_destination}"
        )

    while True:
        raw = input("\nSelect an agent id (or press Enter for 1): ").strip()
        if raw == "":
            return agent_list[0]

        try:
            aid = int(raw)
            for agent in agent_list:
                if agent.agent_id == aid:
                    return agent
        except ValueError:
            pass

        print("Invalid agent id. Try again.")

def interactive_demo(agent_list: List[AgentSnapshot]) -> None:
    print("\n" + "=" * 70)
    print("INTERACTIVE CLICKED-AGENT NLP DEMO")
    print("=" * 70)

    agent = choose_agent_cli(agent_list)

    print("\nSelected agent:")
    print({
        "agent_id": agent.agent_id,
        "student_type": agent.student_type,
        "sport": agent.sport,
        "day": agent.day,
        "time_slot": agent.time_slot,
        "current_location": agent.current_location,
        "next_destination": agent.next_destination,
        "target_location_now": agent.target_location_now
    })

    print("\nAsk supported questions such as:")
    print("- Where are you going?")
    print("- Why are you going there?")
    print("- Why are you here?")
    print("- Are you in the right place right now?")
    print("- Where are you likely to be next?")
    print("- What is the probability you go to Track?")
    print("Type 'quit' to stop.")

    while True:
        q = input("\nYour question: ").strip()
        if q.lower() in {"quit", "exit"}:
            print("Demo ended.")
            break

        result = answer_agent_question(agent, q)
        print("\nParsed question:")
        print(result["parsed_question"])
        print("Semantic frame:")
        print(result["semantic_frame"])
        print("Semantic check:")
        print(result["semantic_check"])
        print("Interpretations:")
        for interp in result["interpretations"][:3]:
            print(" ", interp)
        print("Best interpretation:")
        print(" ", result["best_interpretation"])
        print("Answer:")
        print(" ", result["answer"])

# Uncomment to use:
# interactive_demo(agents)

# ============================================================
# 15. Markdown-ready explanation
# ============================================================

print("\n\n--- Markdown-ready explanation (paste into notebook/report) ---\n")
print(r"""
### Module 9 — Natural Language Processing System

This module extends the campus project by allowing a user to select a campus agent and ask natural-language questions about that agent’s current state and likely next behavior.

A key design improvement in this version is the strict separation between **parsing** and **ambiguity resolution**, together with the addition of a **semantic frame** layer between parsing and answer generation.

### Part 1 — Language modeling
A simple smoothed bigram language model is used to assign scores to supported campus-related question forms such as:
- "Where are you going?"
- "Why are you going there?"
- "What is the probability you go to Track?"

### Part 2 — Grammar and parsing
The parser maps each question into a structured representation containing:
- normalized text
- predicted intent
- grammar match
- explicitly mentioned locations
- explicitly mentioned time references

The system then converts this parse into a semantic frame containing:
- query type
- agent reference
- predicate
- location
- time

This creates a richer NLP pipeline than simple intent matching alone.

### Part 3 — Augmented grammar
The parser is augmented with semantic constraints from the campus domain. For example:
- locations must belong to the canonical campus schema
- probability questions require an explicit recognizable location
- certain time windows carry domain meaning in the broader project

### Part 4 — Handling ambiguity
Ambiguity is handled in a separate stage after parsing. If the question contains vague references such as:
- "there"
- "here"
- "4"

the system generates multiple contextual interpretations and ranks them. This keeps the parsing stage clean while still supporting realistic natural-language interaction.

### Part 5 — NLP task
The implemented NLP task is **classification**. Each question is classified into one of the supported intents:
- `next_destination`
- `destination_reason`
- `current_location_reason`
- `next_location_prediction`
- `validity_check`
- `probability_to_location`

### Integration with the semester project
This module uses the movement and policy logic from the reinforcement learning environment. A selected agent snapshot contains:
- current location
- time slot
- target location
- next action from policy
- next destination

The natural-language answer is grounded in that state, which makes the system appropriate for the final project goal of clicking an agent on a campus interface and asking it questions about where it is going and why.

### Summary
This module satisfies the NLP requirements by including:
- raw text processing
- language modeling
- grammar-based parsing
- semantic frame construction
- augmented semantic constraints
- explicit ambiguity handling
- question classification
- grounded answers tied to the selected campus agent
""".strip())





MODULE 9 — NATURAL LANGUAGE PROCESSING SYSTEM

Training policy from Module 8 environment...
Policy training complete.

Example agent snapshots:
{'agent_id': 1, 'student_type': 'Athlete', 'sport': 'TrackAndField', 'day': 'Tuesday', 'time_slot': '15:00-15:30', 'current_location': 'StudentCenter', 'next_destination': 'Track', 'target_location_now': 'Track'}
{'agent_id': 2, 'student_type': 'Athlete', 'sport': 'TrackAndField', 'day': 'Tuesday', 'time_slot': '16:00-17:30', 'current_location': 'VillagePark', 'next_destination': 'VillagePark', 'target_location_now': 'Track'}
{'agent_id': 3, 'student_type': 'Athlete', 'sport': 'Baseball', 'day': 'Monday', 'time_slot': '13:00-14:30', 'current_location': 'WestPenn', 'next_destination': 'Track', 'target_location_now': 'OffCampus'}

DEMO QUERIES

Question: Where are you going?
Parsed: {'raw_question': 'Where are you going?', 'normalized_question': 'where are you going', 'intent': 'next_destination', 'intent_confidence': 0.9999999996, 'grammar_matc

## Reflection Questions

### How did structure improve understanding?

Structure significantly improved understanding by transforming raw natural language into clear, interpretable representations. Instead of directly reacting to user input, the system processes each question through multiple layers: normalization, parsing, semantic frame construction, and interpretation. 

The introduction of **semantic frames** was especially important. By mapping questions into structured components such as `query_type`, `predicate`, `location`, and `time`, the system can reason about user intent in a consistent and explainable way. For example, questions like “Where are you going?” and “Will you be in the center next?” are both understood through structured representations rather than surface wording.

This structured approach also made the system easier to debug and extend. Each stage (parsing, ambiguity handling, answering) is clearly separated, which improves both clarity and reliability.


### Where did ambiguity cause problems?

Ambiguity caused problems mainly when users used vague terms such as **“there”**, **“here”**, or implicit references to locations or time. For example:
- “Why are you going there?” does not explicitly specify a location.
- “Are you supposed to be here?” depends on what “here” refers to.
- “Will you be in the center next?” requires resolving “center” to `StudentCenter`.

Initially, ambiguity led to incorrect assumptions or missing information. The system could misinterpret intent or fail to extract the correct location.

This was addressed by separating parsing from interpretation and introducing an **ambiguity resolution stage**. The system now generates multiple interpretations (e.g., “there” = next destination vs. current location) and ranks them. This allows the system to handle uncertainty more realistically rather than forcing a single guess during parsing.


### How did probabilistic methods help?

Probabilistic methods helped in two main ways:

1. **Language Modeling**  
   A smoothed bigram language model assigns log-probabilities to sentences. This provides a quantitative measure of how plausible a question is based on learned patterns. While simple, it demonstrates how probabilistic models can evaluate natural language inputs.

2. **Decision and Behavior Modeling (from previous modules)**  
   The system uses reinforcement learning to determine agent behavior. The predicted next destination of an agent is based on learned policies rather than fixed rules. This introduces a probabilistic understanding of movement and decision-making across the campus environment.

3. **Probability Estimates in Answers**  
   The system provides heuristic probability estimates (e.g., likelihood of going to a location next) based on current state, target location, and policy output. While not fully learned probabilities, they still reflect uncertainty and decision tendencies.

Overall, probabilistic methods allowed the system to move beyond deterministic logic and incorporate uncertainty in both language interpretation and agent behavior.


### What limitations remain in your system?

Several limitations remain in the current system:

1. **Language Model Simplicity**  
   The bigram language model is basic and does not capture deeper linguistic structure or long-range dependencies. It assigns probabilities but does not strongly influence interpretation decisions.

2. **Rule-Based Parsing**  
   The grammar and parsing system is primarily pattern-based rather than a fully formal syntactic parser. While effective for this domain, it may not generalize well to more complex or unexpected sentence structures.

3. **Limited Vocabulary and Expressions**  
   The system only supports a predefined set of intents and phrasings. More natural or varied user inputs may not be correctly classified or parsed.

4. **Heuristic Probability Estimates**  
   The probability responses are not learned from data but are based on heuristics tied to the agent’s current state and policy. This limits their accuracy and realism.

5. **No Deep Context or Memory**  
   The system does not maintain conversational context across multiple questions. Each query is processed independently.

6. **Simplified Environment Representation**  
   The campus is modeled at a high level (main locations only), and agent movement is simplified. Real-world behaviors would involve more complex spatial and temporal dynamics.

Despite these limitations, the system successfully demonstrates a full NLP pipeline integrated with a decision-making environment, which satisfies the project requirements.